In [1]:
import jax
import jax.numpy as jnp
import jax.random as jrandom

import time
import os
from IPython.display import display

jax.default_backend()
key = jrandom.key(1)

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

In [2]:
N = 64_000
DIM = 8192

# Generates N Vectors with dimension DIM 
batch = jrandom.normal(key, shape=(N, DIM))

In [3]:
def batch_norm_eager_jax():

    # Begin Timer
    begin = time.perf_counter()

    # Run the Batch Norm Computation in eager mode
    for i in range(N):
        l2norm = jnp.linalg.norm(batch[i])
        l2norm.block_until_ready()
        # do something with the norm

    # End Timer
    end = time.perf_counter()
    
    # Metrics 
    display(f"Elapsed Time = {end - begin}")

def batch_norm_vmap_jax():

    # Begin Timer
    begin = time.perf_counter()

    # Run Batch Norm Computation Jax vmap function
    jax.vmap(jnp.linalg.norm, in_axes=0)(batch).block_until_ready()

    # End Timer
    end = time.perf_counter()

    # Metrics
    display(f"Elapsed Time = {end - begin}")

def batch_norm_vmap_jax_jitted():

    # Internal Function for Jitting
    def batch_norm_vmap_jax_internal(batch):
        return jax.vmap(jnp.linalg.norm, in_axes=0)(batch)
    
    # Jitted Function
    jitted_vmap_internal = jax.jit(batch_norm_vmap_jax_internal)

    # Warmup
    jitted_vmap_internal(batch[:10]).block_until_ready()

    # Begin Timer
    begin = time.perf_counter()
    # Run Batch Norm on Jitted Function
    jitted_vmap_internal(batch).block_until_ready()
    # End Timer
    end = time.perf_counter()
    # Metrics
    display(f"Elapsed Time = {end - begin}")

batch_norm_eager_jax()
batch_norm_vmap_jax()
batch_norm_vmap_jax_jitted()

'Elapsed Time = 8.978008628997486'

'Elapsed Time = 0.052100048000284005'

'Elapsed Time = 0.05554668100376148'

In [ ]:
# Unintuitive Example, Gradient Calculation for each point in the grid

GRID = 150

xcoordinates = jnp.arange(-GRID, GRID, 1)
ycoordinates = jnp.arange(-GRID, GRID, 1)
zcoordinates = jnp.arange(-GRID, GRID, 1)

xx, yy, zz = jnp.meshgrid(xcoordinates, ycoordinates, zcoordinates)

points = jnp.vstack([xx.ravel(), yy.ravel(), zz.ravel()]).astype(jnp.float32)

def scalar_field(point):
    x, y, z = point[0], point[1], point[2]
    return jnp.sin(x) + jnp.exp(y + z)

gradient_at_a_point = jax.grad(scalar_field)
gradient_at_a_point_vmap = jax.vmap(gradient_at_a_point, in_axes=0)


In [13]:
def benchmark_gradient_at_a_point_eager():
    # Begin Timer
    begin = time.perf_counter()
    for point in points:
        jax.grad(scalar_field)(point)
    # End Timer
    end = time.perf_counter()
    # Metrics
    display(f"Elapsed Time = {end - begin}")


def benchmark_gradient_at_a_point_vmap():
    # Begin Timer
    begin = time.perf_counter()
    gradient_at_a_point_vmap(points).block_until_ready()
    # End Timer
    end = time.perf_counter()
    # Metrics
    display(f"Elapsed Time = {end - begin}")

benchmark_gradient_at_a_point_eager()
benchmark_gradient_at_a_point_vmap()

'Elapsed Time = 0.03238164800131926'

'Elapsed Time = 0.025443432998145'